# Hafta 2 — Algoritma Analizi ve Big-O Gösterimi
**Veri Yapıları ve Algoritmalar (YBS)**

Bu not defteri, algoritmaların nasıl ölçeklendiği üzerine sadece düşünmenizi değil, ölçmenizi de sağlar.

## 1. O(1), O(n), O(n²): Doğrudan Zamanlamak

In [ ]:
import time

def ilkini_getir(ogeler):
    return ogeler[0]

def dogrusal_arama(ogeler, hedef):
    for x in ogeler:
        if x == hedef:
            return True
    return False

def yinelenen_cift_var_mi(ogeler):
    for i in range(len(ogeler)):
        for j in range(i + 1, len(ogeler)):
            if ogeler[i] == ogeler[j]:
                return True
    return False

In [ ]:
boyutlar = [1000, 2000, 4000]
for n in boyutlar:
    veri = list(range(n))

    baslangic = time.perf_counter()
    ilkini_getir(veri)
    t_sabit = time.perf_counter() - baslangic

    baslangic = time.perf_counter()
    dogrusal_arama(veri, -1)  # en kötü durum: bulunamadı
    t_dogrusal = time.perf_counter() - baslangic

    baslangic = time.perf_counter()
    yinelenen_cift_var_mi(veri)  # en kötü durum: yinelenen yok
    t_karesel = time.perf_counter() - baslangic

    print(f"n={n:>5} | O(1): {t_sabit:.6f}s | O(n): {t_dogrusal:.6f}s | O(n^2): {t_karesel:.6f}s")

O(1) süresinin neredeyse hiç değişmediğine, O(n) süresinin n ile kabaca orantılı büyüdüğüne ve O(n²) süresinin çok daha hızlı büyüdüğüne dikkat edin — n'i iki katına çıkarmak onu kabaca dört katına çıkarır.

## 2. İş Senaryosu: İki Satış Raporu Algoritması
Geliştirici A'nın O(n²) benzeri yaklaşımını Geliştirici B'nin O(n) yaklaşımıyla karşılaştırın.

In [ ]:
import random

random.seed(0)
urunler = [f"P{i}" for i in range(50)]
islemler = [{"urun": random.choice(urunler), "tutar": round(random.uniform(10, 500), 2)}
            for _ in range(20000)]

def gelistirici_a(islemler, urunler):
    toplamlar = {}
    for urun in urunler:                     # p ürün
        toplam = 0
        for i in islemler:                   # ürün başına n işlem taranır
            if i["urun"] == urun:
                toplam += i["tutar"]
        toplamlar[urun] = toplam
    return toplamlar

def gelistirici_b(islemler):
    toplamlar = {}
    for i in islemler:                        # tek geçiş, O(n)
        toplamlar[i["urun"]] = toplamlar.get(i["urun"], 0) + i["tutar"]
    return toplamlar

baslangic = time.perf_counter(); gelistirici_a(islemler, urunler); print("Geliştirici A:", time.perf_counter() - baslangic, "s")
baslangic = time.perf_counter(); gelistirici_b(islemler); print("Geliştirici B:", time.perf_counter() - baslangic, "s")

## 3. İkili Arama — O(log n)

In [ ]:
def ikili_arama(sirali_idler, hedef):
    alt, ust = 0, len(sirali_idler) - 1
    adim = 0
    while alt <= ust:
        adim += 1
        orta = (alt + ust) // 2
        if sirali_idler[orta] == hedef:
            return orta, adim
        elif sirali_idler[orta] < hedef:
            alt = orta + 1
        else:
            ust = orta - 1
    return -1, adim

sirali_idler = list(range(0, 1_000_000, 2))  # 500.000 öğe
indeks, adim = ikili_arama(sirali_idler, 999998)
print(f"500.000 öğe içinde sadece {adim} adımda {indeks} indeksinde bulundu")

## 4. Zaman-Alan Ödünleşimi: Yeniden Hesaplama ve İndeksleme

In [ ]:
def toplam_satis_yeniden_hesapla(islemler, urun):
    return sum(i["tutar"] for i in islemler if i["urun"] == urun)

def satis_indeksi_olustur(islemler):
    indeks = {}
    for i in islemler:
        indeks[i["urun"]] = indeks.get(i["urun"], 0) + i["tutar"]
    return indeks

# Yeniden hesaplama: bellekte ucuz, çok kez çağrılırsa pahalı
baslangic = time.perf_counter()
for _ in range(20):
    toplam_satis_yeniden_hesapla(islemler, "P1")
print("20 yeniden hesaplama:", time.perf_counter() - baslangic, "s")

# Bir kez indeksle, sonra O(1) aramalar
baslangic = time.perf_counter()
indeks = satis_indeksi_olustur(islemler)
for _ in range(20):
    indeks["P1"]
print("Bir kez oluştur + 20 arama:", time.perf_counter() - baslangic, "s")

## 5. Alıştırmalar
1. Tek döngülü bir fonksiyon yazın ve Big-O'sunu belirleyip yorum olarak yazın.
2. Aynı liste üzerinde iki iç içe döngüsü olan bir fonksiyon yazın ve Big-O'sunu belirleyin.
3. `time.perf_counter()` kullanarak, Hafta 1'deki `siparisi_isle` fonksiyonunuzun çalışma süresinin sepetteki öğe sayısı 5'ten 5.000'e çıkarken nasıl değiştiğini ölçün (tüm ürünlerin stokta olduğunu varsayın).
4. Bir markdown hücresinde, slaytlardaki İK "yönetici arama" probleminin yeniden hesaplama yaklaşımını mı yoksa indeksli yaklaşımı mı tercih etmesi gerektiğini ve nedenini açıklayın.

In [ ]:
# Çözümünüzü buraya yazın


## Özet
Big-O, ölçeklenme davranışını sadece sonradan gözlemlemek yerine önceden tahmin etmemizi sağlar. Yaygın sınıflar — O(1), O(log n), O(n), O(n log n), O(n²) — bu dersin kuracağı neredeyse her algoritmayı kapsar. **Gelecek hafta:** somut bir veri yapısına ilk derinlemesine bakışımız — diziler ve listeler.